# sum-and-broadcast-duality — worked example 2: broadcast_back: sum out the expanded axes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sum-and-broadcast-duality`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The dual of `sum_back` is `broadcast_back`: the backward of a broadcast sums the gradient over the axes that were expanded, so the gradient shape collapses back to `x.shape`. First peel leading axes that broadcasting added, then sum (keepdim) over size-1 axes that were expanded.

## Worked solution

The forward broadcasts `x` of shape `(1, 3)` to `(4, 3)`. In `broadcast_back` we first peel any leading axes (none here since ndim matches), then for each axis where `x.shape[i] == 1` but the gradient is larger, we `sum(dim=i, keepdim=True)`. Here axis 0 was expanded from 1 to 4, so we sum it out with keepdim, giving back `(1, 3)`. We confirm the result matches autograd on `x.broadcast_to((4,3))` and print the gradient shape.

In [ ]:
Tensor = t.Tensor


def broadcast_back(grad_out, out, x):
    while grad_out.ndim > x.ndim:
        grad_out = grad_out.sum(dim=0)
    for i, size in enumerate(x.shape):
        if size == 1 and grad_out.shape[i] != 1:
            grad_out = grad_out.sum(dim=i, keepdim=True)
    return grad_out


t.manual_seed(0)
x = t.randn(1, 3)
out = x.broadcast_to(4, 3)
grad_out = t.randn(4, 3)
grad_x = broadcast_back(grad_out, out, x)
print('grad_x shape:', tuple(grad_x.shape))

xg = x.clone().requires_grad_(True)
xg.broadcast_to(4, 3).backward(grad_out)
print('matches autograd:', bool(t.allclose(grad_x, xg.grad)))